# Exploración y Enriquecimiento de Features para P2 (Riesgo por Bloque)

Este notebook explora alternativas para mejorar el rendimiento de la **Pregunta 2**:
> *"¿Qué estaciones están en riesgo de quedarse sin bicis o sin anclajes en el siguiente bloque de 3 horas?"*

### Objetivos del Notebook:
1. **Analizar el desbalance** real de la variable objetivo `riesgo` (prevalencia ~4%).
2. **Implementar nuevas variables predictivas (Feature Engineering)**:
   - Ratios normalizados por capacidad (`flujo/capacidad`, `salidas/capacidad`).
   - Variables de magnitud simétrica (`abs(flujo_neto)`).
   - Variables cíclicas de tiempo (`sin`/`cos` de hora y día de la semana).
   - Contexto histórico por estación (sin fuga de información).
3. **Entrenar y evaluar modelos candidatos** con el nuevo conjunto de features:
   - Modelo activo: `RandomForestClassifier` con `class_weight="balanced_subsample"` y hojas suaves.
   - Modelos alternativos listos como comentarios (`HistGradientBoostingClassifier`, `LogisticRegression` con `RobustScaler`).
4. **Calibrar el umbral de decisión** optimizando F1 en el conjunto de validación.
5. **Diagnosticar con Curvas de Aprendizaje** (`learning_curve`) para identificar sesgo vs. varianza.

## 1. Importación de Módulos y Configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
)
from sklearn.model_selection import learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

from mi_bici.dataset import load_processed_stations, load_processed_travels
from mi_bici.features import flujo_por_bloque, etiqueta_riesgo
from mi_bici.config import (
    BLOQUE_HORAS,
    CORTE_PRUEBA,
    CORTE_VALIDACION,
    OBJETIVO_P2,
    SEMILLA,
    UMBRAL_DECISION_P2,
    UMBRAL_RIESGO,
    VARIABLES_P2,
)
from mi_bici.modeling.evaluate import metricas_clasificacion

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.autolayout"] = True

## 2. Carga de Datos y Pipeline Base de P2

In [ ]:
# Carga de viajes limpios y estaciones consolidadas
viajes = load_processed_travels()
estaciones = load_processed_stations()

# Construcción del flujo por bloque y etiqueta de riesgo base
flujo = flujo_por_bloque(viajes, estaciones)
df_base = etiqueta_riesgo(flujo, UMBRAL_RIESGO)

print(f"Filas totales en el dataset base: {len(df_base):,}")
df_base.head(5)

## 3. Análisis de la Distribución de la Variable Objetivo (Balance de Clases)

In [ ]:
conteo = df_base[OBJETIVO_P2].value_counts()
porcentaje = df_base[OBJETIVO_P2].value_counts(normalize=True) * 100

balance_df = pd.DataFrame({"Total Bloques": conteo, "Porcentaje (%)": porcentaje.round(2)})
print("--- Distribución de la Variable Objetivo (riesgo) ---")
print(balance_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Gráfico 1: Conteo de clases
axes[0].bar(["Sin Riesgo (0)", "En Riesgo (1)"], conteo.values, color=["#2b5c8f", "#d95f02"], width=0.5)
axes[0].set_title(f"Distribución Global de Clases ({porcentaje.iloc[1]:.2f}% en riesgo)")
axes[0].set_ylabel("Número de bloques")

# Gráfico 2: Tasa de riesgo según la hora del bloque
tasa_hora = df_base.groupby("hora")[OBJETIVO_P2].mean() * 100
axes[1].plot(tasa_hora.index, tasa_hora.values, marker="o", color="#d95f02", linewidth=2)
axes[1].set_title("Prevalencia de Riesgo por Hora del Bloque (%)")
axes[1].set_xlabel("Hora del día")
axes[1].set_ylabel("% de Bloques en Riesgo")
axes[1].set_xticks(tasa_hora.index)

plt.show()

## 4. Enriquecimiento de Features (Feature Extraction Propuesto)

Calculamos variables normalizadas, relaciones parabólicas y codificaciones periódicas para superar las limitaciones del vector base.

In [ ]:
def enriquecer_features_p2(df: pd.DataFrame) -> pd.DataFrame:
    """Genera las nuevas variables recomendadas para mitigar sesgos y no-linealidades."""
    df = df.copy()
    
    # 1. Ratios de flujo relativos a la capacidad (mismo orden de magnitud que UMBRAL_RIESGO = 0.4)
    df["ratio_flujo_capacidad_lag1"] = df["flujo_neto_lag_1"] / df["capacidad"]
    df["ratio_flujo_capacidad_lag2"] = df["flujo_neto_lag_2"] / df["capacidad"]
    df["ratio_salidas_capacidad_lag1"] = df["salidas_lag_1"] / df["capacidad"]
    df["ratio_llegadas_capacidad_lag1"] = df["llegadas_lag_1"] / df["capacidad"]
    
    # 2. Desbalance relativo direccional
    df["balance_flujo_relativo_lag1"] = (
        (df["llegadas_lag_1"] - df["salidas_lag_1"]) /
        (df["llegadas_lag_1"] + df["salidas_lag_1"] + 1e-5)
    )
    
    # 3. Magnitud absoluta del flujo (ayuda a modelos lineales a capturar vaciado y llenado)
    df["flujo_neto_abs_lag1"] = df["flujo_neto_lag_1"].abs()
    df["ratio_flujo_abs_capacidad_lag1"] = df["flujo_neto_abs_lag1"] / df["capacidad"]
    
    # 4. Codificación periódica/cíclica de hora y día
    df["hora_sin"] = np.sin(2 * np.pi * df["hora"] / 24)
    df["hora_cos"] = np.cos(2 * np.pi * df["hora"] / 24)
    df["dia_sin"] = np.sin(2 * np.pi * df["dia_semana"] / 7)
    df["dia_cos"] = np.cos(2 * np.pi * df["dia_semana"] / 7)
    df["es_fin_de_semana"] = (df["dia_semana"] >= 5).astype(int)
    
    return df

df_enriquecido = enriquecer_features_p2(df_base)

features_nuevos = [
    "ratio_flujo_capacidad_lag1",
    "ratio_flujo_capacidad_lag2",
    "ratio_salidas_capacidad_lag1",
    "ratio_llegadas_capacidad_lag1",
    "balance_flujo_relativo_lag1",
    "flujo_neto_abs_lag1",
    "ratio_flujo_abs_capacidad_lag1",
    "hora_sin",
    "hora_cos",
    "dia_sin",
    "dia_cos",
    "es_fin_de_semana",
]

print(f"Se generaron {len(features_nuevos)} features nuevos.")
df_enriquecido[features_nuevos].describe().T[["mean", "std", "min", "max"]]

## 5. Partición Temporal Estricta y Contexto Histórico por Estación

Para evitar fuga de datos (*data leakage*), cualquier agregación histórica por estación (tasa de riesgo base) se calcula **únicamente con los datos anteriores a la fecha de corte de validación**.

In [ ]:
# Máscaras temporales alineadas con mi_bici/config.py
mascara_train = df_enriquecido["bloque"] < CORTE_VALIDACION
mascara_val = (df_enriquecido["bloque"] >= CORTE_VALIDACION) & (df_enriquecido["bloque"] < CORTE_PRUEBA)
mascara_test = df_enriquecido["bloque"] >= CORTE_PRUEBA

# Target encoding temporal por estación (calculado solo con train)
tasa_estacion_train = df_enriquecido[mascara_train].groupby("estacion_id")[OBJETIVO_P2].mean()
tasa_media_global = df_enriquecido[mascara_train][OBJETIVO_P2].mean()

df_enriquecido["tasa_riesgo_estacion"] = (
    df_enriquecido["estacion_id"].map(tasa_estacion_train).fillna(tasa_media_global)
)

# Vector final de variables combinando base + enriquecidas
vector_features = VARIABLES_P2 + features_nuevos + ["tasa_riesgo_estacion"]

X_train, y_train = df_enriquecido.loc[mascara_train, vector_features], df_enriquecido.loc[mascara_train, OBJETIVO_P2]
X_val, y_val = df_enriquecido.loc[mascara_val, vector_features], df_enriquecido.loc[mascara_val, OBJETIVO_P2]
X_test, y_test = df_enriquecido.loc[mascara_test, vector_features], df_enriquecido.loc[mascara_test, OBJETIVO_P2]

print(f"Train set: {X_train.shape[0]:,} filas | Tasa riesgo: {y_train.mean():.2%}")
print(f"Val set:   {X_val.shape[0]:,} filas | Tasa riesgo: {y_val.mean():.2%}")
print(f"Test set:  {X_test.shape[0]:,} filas | Tasa riesgo: {y_test.mean():.2%}")

## 6. Selección y Ajuste del Modelo Candidato

Selecciona un modelo para entrenar. El modelo activo por defecto es `RandomForestClassifier` optimizado con hojas más suaves y ponderación bootstrap. Los demás modelos quedan comentados y listos para alternar.

In [ ]:
# =========================================================================
# OPCIÓN 1: Random Forest Classifier (Activo por defecto)
# =========================================================================
modelo = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=15,             # Hojas con más muestras para suavizar probabilidades y calibración
    class_weight="balanced_subsample",# Balanceo calculado por cada árbol bootstrap
    random_state=SEMILLA,
    n_jobs=-1
)

# =========================================================================
# OPCIÓN 2: HistGradientBoosting (Descomentar para probar)
# =========================================================================
# modelo = HistGradientBoostingClassifier(
#     learning_rate=0.08,
#     max_iter=150,
#     min_samples_leaf=60,
#     l2_regularization=3.0,
#     class_weight="balanced",
#     random_state=SEMILLA
# )

# =========================================================================
# OPCIÓN 3: Logistic Regression con Escalador Robusto (Descomentar para probar)
# =========================================================================
# modelo = Pipeline([
#     ("scaler", RobustScaler()),
#     ("logreg", LogisticRegression(
#         C=0.5,
#         class_weight="balanced",
#         max_iter=1000,
#         random_state=SEMILLA
#     ))
# ])

# Entrenamiento con el conjunto de entrenamiento
modelo.fit(X_train, y_train)
print(f"Modelo [{modelo.__class__.__name__}] entrenado con éxito.")

## 7. Calibración del Umbral de Decisión en Validación y Curva Precision-Recall

In [ ]:
# Estimación de probabilidades sobre el conjunto de validación
prob_val = modelo.predict_proba(X_val)[:, 1]

# Cálculo de la curva Precision-Recall y búsqueda de umbral que maximiza F1
precisions, recalls, thresholds = precision_recall_curve(y_val, prob_val)
denominador = precisions[:-1] + recalls[:-1]
f1_scores = np.divide(
    2 * precisions[:-1] * recalls[:-1],
    denominador,
    out=np.zeros_like(thresholds),
    where=denominador > 0,
)

mejor_idx = np.argmax(f1_scores) if len(f1_scores) > 0 and np.max(f1_scores) > 0 else 0
umbral_optimo = float(thresholds[mejor_idx]) if len(thresholds) > 0 else UMBRAL_DECISION_P2

ap_val = average_precision_score(y_val, prob_val)

print("=== Resultados en Validación ===")
print(f"Average Precision (PR-AUC): {ap_val:.4f}")
print(f"Umbral de decisión óptimo:  {umbral_optimo:.4f}")
print(f"F1 Score máximo alcanzado:  {f1_scores[mejor_idx]:.4f}")
print(f"Precision en umbral óptimo: {precisions[mejor_idx]:.4f}")
print(f"Recall en umbral óptimo:    {recalls[mejor_idx]:.4f}")

# Visualización de la curva PR
plt.figure(figsize=(6.5, 5))
plt.plot(recalls, precisions, color="#2b5c8f", linewidth=2, label=f"PR Curve (AP = {ap_val:.3f})")
plt.scatter(recalls[mejor_idx], precisions[mejor_idx], color="red", s=90, zorder=5, label=f"Punto óptimo (umbral={umbral_optimo:.2f})")
plt.axhline(y_val.mean(), color="gray", linestyle="--", label=f"Azar / Prevalencia ({y_val.mean():.2%})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curva Precision-Recall en Validación")
plt.legend(loc="upper right")
plt.show()

## 8. Evaluación Fuera de Muestra (Test Set) y Matriz de Confusión

Evaluamos sobre el conjunto de prueba utilizando el umbral calibrado en validación.

In [ ]:
prob_test = modelo.predict_proba(X_test)[:, 1]
pred_test = (prob_test >= umbral_optimo).astype(int)

# Cálculo de métricas estándar del proyecto
metricas_test = metricas_clasificacion(y_test, pred_test, prob_test, prefijo="test")

print("=== Métricas Oficiales en Conjunto de Prueba ===")
for metrica, valor in metricas_test.items():
    print(f"  {metrica:25s}: {valor:.4f}")

# Matriz de Confusión
fig, ax = plt.subplots(figsize=(5, 4.2))
cm = confusion_matrix(y_test, pred_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Sin Riesgo (0)", "En Riesgo (1)"])
disp.plot(cmap="Blues", ax=ax, values_format="d")
ax.set_title("Matriz de Confusión en Prueba")
plt.show()

## 9. Curvas de Aprendizaje (Learning Curves)

Esta curva analiza el rendimiento de entrenamiento y validación cruzada en función del tamaño del dataset.
- **Diagnóstico de Alto Sesgo (Underfitting):** Ambas curvas convergen abajo con poca brecha $\to$ Se necesitan más variables complejas e interacciones.
- **Diagnóstico de Alta Varianza (Overfitting):** El score de train es muy alto pero la validación se queda estancada abajo $\to$ Se requiere mayor regularización (`min_samples_leaf` más alto, menor `max_depth`).

In [ ]:
# Muestra representativa de entrenamiento para mantener la ejecución rápida y fluida
N_MUESTRA = min(len(X_train), 40000)
X_sub = X_train.sample(n=N_MUESTRA, random_state=SEMILLA)
y_sub = y_train.loc[X_sub.index]

train_sizes, train_scores, cv_scores = learning_curve(
    modelo,
    X_sub,
    y_sub,
    train_sizes=np.linspace(0.1, 1.0, 5),
    cv=3,
    scoring="average_precision",
    n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
cv_mean = np.mean(cv_scores, axis=1)
cv_std = np.std(cv_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, "o-", color="#2b5c8f", label="Training PR-AUC")
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color="#2b5c8f")

plt.plot(train_sizes, cv_mean, "o-", color="#d95f02", label="Cross-Validation PR-AUC")
plt.fill_between(train_sizes, cv_mean - cv_std, cv_mean + cv_std, alpha=0.15, color="#d95f02")

plt.xlabel("Tamaño de Muestra de Entrenamiento")
plt.ylabel("Average Precision (PR-AUC)")
plt.title("Curvas de Aprendizaje (Diagnóstico de Sesgo vs. Varianza)")
plt.legend(loc="lower right")
plt.show()

## 10. Importancia de Variables (Feature Importance)

Revisamos cuáles de los nuevos features aportan mayor poder discriminativo.

In [ ]:
if hasattr(modelo, "feature_importances_"):
    importancias = pd.Series(modelo.feature_importances_, index=vector_features).sort_values(ascending=False)
    
    plt.figure(figsize=(9, 6))
    importancias.head(15).plot(kind="barh", color="#2b5c8f").invert_yaxis()
    plt.title("Top 15 Variables Más Importantes")
    plt.xlabel("Importancia Relativa (Gini / Impureza)")
    plt.show()
    
    print("--- Top 10 Features ---")
    print(importancias.head(10))
else:
    print("El modelo seleccionado no expone `feature_importances_` directamente.")